# MLOps Platform — Exploratory Data Analysis & Model Exploration

Use this notebook to:
- Explore training data distributions
- Visualize drift between reference and current data
- Manually run and compare model experiments
- Inspect model registry contents

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

plt.style.use('dark_background')
np.random.seed(42)
print('✅ Imports ready')

## 1. Generate & Explore Training Data

In [ ]:
feature_names = [
    'age_normalized', 'income_normalized', 'credit_score_normalized',
    'loan_amount_normalized', 'employment_years', 'debt_ratio'
]

# Reference (training) distribution
n = 5000
X_ref = np.random.randn(n, 6)
y = (0.3*X_ref[:,0] + 0.4*X_ref[:,1] - 0.2*X_ref[:,2] + np.random.randn(n)*0.3) > 0

df = pd.DataFrame(X_ref, columns=feature_names)
df['label'] = y.astype(int)

print(df.describe().round(3))
print(f'\nClass balance: {df.label.value_counts().to_dict()}')

## 2. Feature Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#06b6d4']

for idx, (feat, ax, color) in enumerate(zip(feature_names, axes.flat, colors)):
    ax.hist(df[feat], bins=40, color=color, alpha=0.75, edgecolor='none')
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('Feature Distributions — Training Data', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../docs/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Simulate & Visualize Data Drift

In [ ]:
# Simulate drifted data (mean shift on some features)
drift_magnitudes = [0.0, 0.5, 0.0, 1.2, 0.0, 0.3]
X_drifted = np.random.randn(500, 6)
for i, drift in enumerate(drift_magnitudes):
    X_drifted[:, i] += drift

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for idx, (feat, ax) in enumerate(zip(feature_names, axes.flat)):
    ax.hist(X_ref[:, idx], bins=30, alpha=0.6, label='Reference', color='#3b82f6')
    ax.hist(X_drifted[:, idx], bins=30, alpha=0.6, label='Current', color='#ef4444')
    
    ks_stat, p_val = stats.ks_2samp(X_ref[:, idx], X_drifted[:, idx])
    drifted = p_val < 0.05
    ax.set_title(f'{feat}\nKS={ks_stat:.3f}, p={p_val:.3f} {"⚠️" if drifted else "✓"}', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Drift Analysis: Reference vs Current Data', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../docs/drift_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Features with significant drift (p < 0.05):', 
      [feature_names[i] for i in range(6) if stats.ks_2samp(X_ref[:,i], X_drifted[:,i])[1] < 0.05])

## 4. Train & Compare Models

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X_ref, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
for name, clf in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))

## 5. Model Registry Inspection

In [ ]:
import asyncio
import os
os.chdir('..')

from src.models.model_registry import ModelRegistry

async def inspect_registry():
    registry = ModelRegistry()
    await registry.initialize()
    models = await registry.list_models()
    print(f'Total models in registry: {len(models)}')
    for m in models:
        print(f"  [{m['status'].upper():12}] {m['model_id']}  acc={m['accuracy']}  f1={m['f1_score']}")
    return models

models = asyncio.run(inspect_registry())